# [실습] 전처리 파이프라인과 데이터 누수 — 점수를 믿어도 되는가

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# Chapter 4 실습. 이 점수를 믿어도 되는가?

**필수 머신러닝 · Chapter 4**

| Part | 주제 |
| --- | --- |
| Part 1 | 너무 좋은 점수 의심하기 |
| Part 2 | 전처리를 올바른 순서로 |
| Part 3 | 점수를 믿는 법: 교차검증과 시간 순 분할 |
| Part 4 | 보고할 숫자 정하기 |

## 분석 의뢰서

> **의뢰 부서**: 은행 마케팅팀 (Chapter 3의 후속 의뢰)
> **목적**: 지난번 모델을 다음 캠페인에 실제로 쓰려 한다. 그런데 팀원 한 명이 통화 시간 변수를 추가했더니 점수가 훨씬 좋아졌다고 한다.
> **제약**: 모델이 약속한 가입 수가 실제와 크게 다르면 캠페인 예산 승인을 받을 수 없다.
> **질문**: 어느 모델의 점수를 믿고 보고해야 하는가? 그 점수가 실제 캠페인에서도 나온다고 어떻게 확신하는가?

## 이 실습의 질문

> **통화 시간을 뺀 모델과 넣은 모델, 무작위 분할과 시간 순 분할 — 마케팅팀에 "다음 캠페인에서 기대할 가입 수"로 보고할 숫자는 어느 것인가? 그리고 그 숫자를 믿을 수 있다고 어떻게 설득할 것인가?**

이 실습에는 새 모델이 나오지 않습니다. Chapter 3의 모델을 그대로 쓰면서, 그 점수를 **의심하고, 고쳐 만들고, 다시 재는** 연습을 합니다.

## 이 노트북을 쓰는 법

| 표시 | 뜻 |
| --- | --- |
| **[직접 작성]** | 빈 코드 셀을 직접 채웁니다. 강의 교안의 코드를 보지 않고 먼저 써 보세요 |
| **[판단]** | 코드가 아니라 생각을 적는 칸입니다. 셀을 더블클릭해 `나의 답:` 뒤에 적으세요 |
| **[제공]** | 미리 채워 둔 코드입니다. 읽고 실행하세요 |
| **힌트** | 막혔을 때만 펼치세요 |
| **먼저 예상하기** | 결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다 |

## 준비

Chapter 3과 같은 `bank-full.csv`(통화 45,211건)를 이 노트북과 같은 폴더에 두세요. 이번에는 Chapter 3에서 뺐던 통화 시간(`duration`) 열을 **지우지 않고** 둡니다.

In [1]:
# [제공] 이 실습에서 쓰는 라이브러리
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# 그래프의 한글 글꼴 (환경에 맞는 글꼴이 없으면 글자가 깨질 수 있습니다)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

---
# Part 1. 너무 좋은 점수 의심하기

## 1-1. 준비: Chapter 3의 데이터 다시 만들기

**[제공]** Chapter 3에서 한 준비를 한 번에 합니다. 데이터를 불러와 `target`을 만들고, 같은 방법(80 : 20, `random_state=42`, `stratify`)으로 나눕니다. 이번에는 `duration` 열을 지우지 않습니다.

| 이름 | 뜻 |
| --- | --- |
| `num_cols` | 숫자 피처 4개: `age`, `balance`, `campaign`, `previous` |
| `cat_cols` | 범주 피처 8개: `poutcome`, `housing`, `loan`, `contact`, `month`, `job`, `marital`, `education` |
| `features` | 두 목록을 합친 Chapter 3 모델의 피처 12개 |

In [2]:
# [제공] 데이터 준비 (Chapter 3과 같음, duration은 남겨 둠)
df = pd.read_csv("bank-full.csv", sep=";")
df["target"] = (df["y"] == "yes").astype(int)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])

num_cols = ["age", "balance", "campaign", "previous"]
cat_cols = ["poutcome", "housing", "loan", "contact", "month", "job", "marital", "education"]
features = num_cols + cat_cols
y_train, y_test = train_df["target"], test_df["target"]
print(len(train_df), len(test_df), "| 평가용 가입률", round(y_test.mean(), 3))

36168 9043 | 평가용 가입률 0.117


### 실습 도구 (실행만 하세요)

| 도구 | 하는 일 |
| --- | --- |
| `record(이름, 정답, 확률, 분할)` | 점수를 결과표에 남긴다: ROC AUC, PR AUC, 평가 데이터의 가입률, 상위 1,000명 중 가입 수, 무작위로 1,000명을 골랐을 때의 기대 가입 수, 배수 |
| `show_log()` | 결과표를 보여 준다 |

In [3]:
# [제공] 실습 도구
results = []

def record(name, y_true, prob, split="무작위"):
    y_true = np.asarray(y_true); prob = np.asarray(prob)
    top = y_true[np.argsort(-prob, kind="stable")][:1000].sum()
    base = y_true.mean()
    row = {"실험": name, "분할": split,
           "ROC AUC": round(roc_auc_score(y_true, prob), 3),
           "PR AUC": round(average_precision_score(y_true, prob), 3),
           "평가 가입률": round(base, 3),
           "상위 1,000명 가입": int(top),
           "무작위 1,000명 기대": round(1000 * base),
           "배수": round(top / (1000 * base), 1)}
    results.append(row)
    print(f"기록: [{name} / {split}] ROC AUC {row['ROC AUC']:.3f} | 상위 1,000명 가입 {row['상위 1,000명 가입']}명 (무작위의 {row['배수']}배)")

def show_log():
    return pd.DataFrame(results)

## 1-2. Chapter 3의 모델 다시 만들기

**[직접 작성]** Chapter 3과 같은 방법으로 로지스틱 회귀를 만들고, 평가용 고객의 가입 확률을 `prob_base`에 저장한 뒤 `record("Chapter 3 모델", y_test, prob_base)`로 기록하세요.

1. `pd.get_dummies(df[features])`로 원핫 인코딩 → 학습용·평가용 행으로 나누기 (`.loc[train_df.index]`)
2. `StandardScaler`를 **학습용으로** `fit`하고, 학습용·평가용을 `transform`
3. `LogisticRegression(max_iter=3000)`으로 학습하고 `predict_proba(...)[:, 1]`

<details><summary>힌트</summary>

```python
X = pd.get_dummies(df[features]).astype(float)
X_tr, X_te = X.loc[train_df.index], X.loc[test_df.index]
scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=3000).fit(scaler.transform(X_tr), y_train)
prob_base = model.predict_proba(scaler.transform(X_te))[:, 1]
```
</details>

In [4]:
# 1-2. Chapter 3 모델 다시 만들기
X = pd.get_dummies(df[features]).astype(float)              # 원핫 인코딩
X_tr, X_te = X.loc[train_df.index], X.loc[test_df.index]    # 학습용 / 평가용 행 나누기

scaler = StandardScaler().fit(X_tr)                         # 기준(평균·표준편차)은 학습용으로만
model = LogisticRegression(max_iter=3000).fit(scaler.transform(X_tr), y_train)
prob_base = model.predict_proba(scaler.transform(X_te))[:, 1]

record("Chapter 3 모델", y_test, prob_base)

기록: [Chapter 3 모델 / 무작위] ROC AUC 0.773 | 상위 1,000명 가입 464명 (무작위의 4.0배)


결과가 Chapter 3과 같은지(ROC AUC 0.773 안팎) 확인하세요.

## 1-3. 통화 시간을 넣어 보면

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. Chapter 3 실습에서 피처를 더하고 설정을 바꿨을 때 ROC AUC가 얼마나 움직였는지 떠올려 보세요.

> **[판단]** 예상: 통화 시간(`duration`, 초)을 피처에 하나 더하면 ROC AUC가 0.773에서 얼마가 될까요?
>
> 나의 답: 0.80 안팎(약 +0.03)으로 예상합니다. Chapter 3에서 피처를 더하거나 설정을 바꿨을 때 ROC AUC는 0.01 안팎으로만 움직였기 때문에, 피처 하나로는 크게 오르지 않을 것 같습니다.

**[직접 작성]** 1-2의 코드에서 피처 목록을 `features + ["duration"]`으로 바꿔 같은 모델을 만들고, 확률을 `prob_dur`에 저장한 뒤 `record("통화 시간 추가", y_test, prob_dur)`로 기록하세요.

In [5]:
# 1-3. 통화 시간(duration)을 피처에 추가
X_dur = pd.get_dummies(df[features + ["duration"]]).astype(float)
X_dur_tr, X_dur_te = X_dur.loc[train_df.index], X_dur.loc[test_df.index]

scaler_dur = StandardScaler().fit(X_dur_tr)
model_dur = LogisticRegression(max_iter=3000).fit(scaler_dur.transform(X_dur_tr), y_train)
prob_dur = model_dur.predict_proba(scaler_dur.transform(X_dur_te))[:, 1]

record("통화 시간 추가", y_test, prob_dur)

기록: [통화 시간 추가 / 무작위] ROC AUC 0.906 | 상위 1,000명 가입 578명 (무작위의 4.9배)


> **[판단]** ROC AUC가 얼마나 올랐나요? 예상과 비교해 보세요. Chapter 3 실습에서 다른 피처를 더했을 때의 변화와 견주면, 이 변화는 어느 정도로 큰가요? 이 결과를 보고 먼저 기뻐해야 할까요, 먼저 의심해야 할까요?
>
> 나의 답: 0.773 → **0.906**으로 약 **0.13** 올랐습니다. 예상(+0.03)보다 4배 이상 큽니다. Chapter 3에서 다른 피처를 더했을 때의 변화(0.01 안팎)와 비교하면 10배가 넘는 변화입니다. 상위 1,000명 가입도 464명 → 578명으로 늘었습니다. 피처 하나로 이 정도로 좋아지는 것은 비정상적이므로 **먼저 의심해야** 합니다. 정답(가입)과 같은 시점이나 그 뒤에 생기는 정보가 섞였을 가능성이 큽니다.

## 1-4. 이 변수는 어디서 오는가

**[직접 작성]** **학습용 데이터**에서 다음 두 가지를 확인하세요.

1. 가입(`target` = 1)과 미가입(0) 고객의 통화 시간 하위 25%, 중앙값, 상위 25%
2. 통화 시간이 60초 미만인 통화와 600초(10분) 초과인 통화의 가입률과 통화 수

<details><summary>힌트</summary>

1번은 `train_df.groupby("target")["duration"].quantile([0.25, 0.5, 0.75]).unstack()`.
2번은 조건을 만들어 `train_df.loc[조건, "target"].agg(["mean", "size"])`.
</details>

In [6]:
# 1-4. 학습용 데이터에서 통화 시간과 가입의 관계 보기
# 1) 가입(1) / 미가입(0) 고객의 통화 시간 분위수
q = train_df.groupby("target")["duration"].quantile([0.25, 0.5, 0.75]).unstack()
q.columns = ["하위 25%", "중앙값", "상위 25%"]
print("통화 시간(초) 분위수")
print(q)
print()

# 2) 아주 짧은 통화(60초 미만)와 아주 긴 통화(600초 초과)의 가입률
short = train_df.loc[train_df["duration"] < 60, "target"].agg(["mean", "size"])
long_ = train_df.loc[train_df["duration"] > 600, "target"].agg(["mean", "size"])
print(f"60초 미만 통화 : 가입률 {short['mean']:.3f} | 통화 수 {int(short['size']):,}")
print(f"600초 초과 통화: 가입률 {long_['mean']:.3f} | 통화 수 {int(long_['size']):,}")

통화 시간(초) 분위수
        하위 25%    중앙값  상위 25%
target                       
0         95.0  164.0   278.0
1        245.0  426.0   734.5

60초 미만 통화 : 가입률 0.002 | 통화 수 3,727
600초 초과 통화: 가입률 0.486 | 통화 수 3,043


> **[판단]** 통화 시간과 가입 사이의 관계는 어느 방향일까요? 통화가 길어서 가입한 것일까요, 가입할 고객이라서 통화가 길어진 것일까요? 마케팅팀이 "누구에게 전화할까"를 정하는 시점에 이 값을 알 수 있나요?
>
> 나의 답: 가입 고객의 통화 시간 중앙값은 426초로, 미가입 고객(164초)의 약 2.6배입니다. 60초 미만 통화의 가입률은 0.2%(3,727건)인 반면, 600초를 넘는 통화는 48.6%(3,043건)입니다. 방향은 **가입할 고객이라서 통화가 길어진 것**에 가깝습니다. 관심 없는 고객은 금방 끊고, 가입하는 고객은 상품 설명과 가입 절차 때문에 통화가 길어집니다. 즉 통화 시간은 가입의 원인이라기보다 결과입니다. 그리고 통화 시간은 **전화가 끝나야** 알 수 있으므로, 마케팅팀이 "누구에게 전화할까"를 정하는 시점에는 **알 수 없습니다.**

## 1-5. 피처마다 예측 시점 점검하기

데이터 설명서에 적힌 각 열의 뜻입니다. 마케팅팀은 **전화를 걸기 전에** 누구에게 걸지 정해야 합니다.

| 열 | 설명서의 뜻 |
| --- | --- |
| `age`, `job`, `marital`, `education` | 고객 정보 |
| `balance`, `housing`, `loan` | 계좌 잔액, 주택 대출, 개인 대출 |
| `contact` | 연락 수단 |
| `month`, `day` | 마지막 통화의 월, 일 |
| `campaign` | 이번 캠페인에서 이 고객에게 건 통화 수 (**마지막 통화 포함**) |
| `pdays`, `previous`, `poutcome` | 이전 캠페인 이후 경과일, 이전 캠페인 통화 수, 이전 캠페인 결과 |
| `duration` | 마지막 통화의 시간 (초) |

> **[판단]** 위 열 중 예측 시점(전화를 걸기 전)에 **알 수 없는** 열과 **애매한** 열을 고르고, 각각 이유를 적으세요. (힌트: `campaign`의 "마지막 통화 포함"을 눈여겨보세요.)
>
> 나의 답: 
>
> **알 수 없는 열**
>
> - `duration`: 마지막 통화가 끝나야 생기는 값입니다. 전화를 걸기 전에는 없습니다.
>
> **애매한 열**
>
> - `campaign`: 이번 캠페인의 통화 수인데 **마지막 통화까지 포함**합니다. 전화를 걸기 전에는 "지금까지 건 횟수"만 알 수 있으니, 1을 빼서 쓰거나 "몇 번째 통화인가"로 다시 정의해야 합니다. 또 가입하면 더 걸지 않으므로 값 자체에 결과가 섞여 있습니다.
> - `month`, `day`: **마지막 통화**의 날짜입니다. 어느 통화가 "마지막"이 될지는 캠페인이 끝나야 정해지므로, 기록 방식에 따라 미래 정보가 될 수 있습니다.
> - `contact`: 이번 캠페인에서 실제로 연결된 연락 수단이라면 통화 후에 확정되는 값일 수 있습니다.
>
> 나머지(고객 정보, 잔액·대출, 이전 캠페인 정보 `pdays`, `previous`, `poutcome`)는 전화를 걸기 전에 알 수 있습니다.

---
# Part 2. 전처리를 올바른 순서로

Part 1의 누수는 **변수**에 숨어 있었습니다. 이 Part의 누수는 **코드의 순서**에 숨어 있습니다. 이제부터는 통화 시간을 쓰지 않고, Chapter 3 모델의 피처 12개(`features`)만 씁니다.

## 2-1. 스케일러의 기준을 전체 데이터로 잡으면

**[직접 작성]** 1-2와 같은 모델을 만들되, 스케일러를 **전체 데이터(학습용 + 평가용)**로 `fit`한 버전의 확률을 `prob_leak`에 저장하세요. 그다음 `prob_base`와 `prob_leak`의 ROC AUC를 소수점 넷째 자리까지 출력해 비교하세요. (이 실험은 결과표에 기록하지 않습니다.)

<details><summary>힌트</summary>

`StandardScaler().fit(X)`처럼 나누기 전의 `X` 전체로 `fit`하면 됩니다. 나머지는 1-2와 같습니다. 출력은 `print(round(roc_auc_score(y_test, prob_leak), 4))`.
</details>

In [7]:
# 2-1. 스케일러 기준을 전체 데이터(학습용 + 평가용)로 잡은 경우
scaler_leak = StandardScaler().fit(X)                       # 나누기 전의 X 전체로 fit (누수)
model_leak = LogisticRegression(max_iter=3000).fit(scaler_leak.transform(X_tr), y_train)
prob_leak = model_leak.predict_proba(scaler_leak.transform(X_te))[:, 1]

print("올바른 방법 (학습용 기준):", round(roc_auc_score(y_test, prob_base), 4))
print("누수 방법   (전체 기준)  :", round(roc_auc_score(y_test, prob_leak), 4))

올바른 방법 (학습용 기준): 0.7729
누수 방법   (전체 기준)  : 0.7729


> **[판단]** 두 점수는 얼마나 다른가요? 왜 이런 결과가 나왔을까요? 점수에 차이가 없다면, 그래도 "기준은 학습용에서만" 규칙을 지켜야 하는 이유는 무엇일까요?
>
> 나의 답: 두 점수는 0.7729로 **똑같습니다**(소수점 넷째 자리까지). 로지스틱 회귀는 피처를 평균·표준편차로 옮기고 늘려도 계수가 그만큼 보정되므로, 기준이 조금 달라져도 확률 순위가 거의 바뀌지 않습니다. 또 학습용(36,168건)이 전체의 80%라서 전체와 학습용의 평균·표준편차가 거의 같습니다. 그래도 규칙을 지켜야 하는 이유는 ① 실제 운영에서는 미래 고객의 데이터가 **아예 없으므로** 그런 기준은 만들 수 없고, ② 평가용 데이터의 분포가 다르거나, 스케일에 민감한 모델(거리 기반 모델 등)에서는 점수가 부풀려질 수 있으며, ③ 2-2처럼 기준에 타깃이 들어가는 전처리에서는 같은 습관이 바로 큰 누수가 되기 때문입니다.

## 2-2. 기준에 타깃이 들어가면: 타깃 인코딩

**타깃 인코딩**은 범주를 "그 범주의 가입률"로 바꾸는 인코딩입니다. 여기서는 통화한 날(월 + 일, 예: `may-12`)을 범주로 써서, 날마다 가입률을 계산해 피처 하나로 만듭니다.

**[제공]** 날짜 범주를 만듭니다.

In [8]:
# [제공] 통화한 날(월-일) 범주
day_key = df["month"] + "-" + df["day"].astype(str)
key_tr, key_te = day_key.loc[train_df.index], day_key.loc[test_df.index]
print("날짜 범주 수:", day_key.nunique(), "| 날짜당 통화 수 중앙값:", int(day_key.value_counts().median()))

날짜 범주 수: 318 | 날짜당 통화 수 중앙값: 29


**[직접 작성]** 날짜별 가입률을 두 가지 방법으로 계산해, 각각 **피처 하나짜리** 로지스틱 회귀를 만들고 평가용 ROC AUC를 비교하세요.

- **올바른 방법**: 가입률을 **학습용 데이터로만** 계산한다 (`train_df["target"].groupby(key_tr).mean()`)
- **누수가 있는 방법**: 가입률을 **전체 데이터로** 계산한다 (`df["target"].groupby(day_key).mean()`)

어느 방법이든 계산한 가입률을 학습용과 평가용 날짜에 `map`으로 붙여 피처로 씁니다. 학습용에 없던 날짜는 학습용 전체 가입률로 채웁니다.

<details><summary>힌트</summary>

```python
rate = train_df["target"].groupby(key_tr).mean()
f_tr = key_tr.map(rate).fillna(y_train.mean()).to_frame()
f_te = key_te.map(rate).fillna(y_train.mean()).to_frame()
m = LogisticRegression().fit(f_tr, y_train)
print(roc_auc_score(y_test, m.predict_proba(f_te)[:, 1]))
```
누수가 있는 방법은 `rate`만 전체 데이터로 바꿉니다.
</details>

In [9]:
# 2-2. 날짜별 가입률(타깃 인코딩) 두 가지 방법 비교
def day_rate_auc(rate):
    """날짜별 가입률(rate)을 피처 하나로 붙여 로지스틱 회귀를 만들고 평가용 ROC AUC를 돌려준다."""
    f_tr = key_tr.map(rate).fillna(y_train.mean()).to_frame()
    f_te = key_te.map(rate).fillna(y_train.mean()).to_frame()
    m = LogisticRegression().fit(f_tr, y_train)
    return roc_auc_score(y_test, m.predict_proba(f_te)[:, 1])

rate_ok = train_df["target"].groupby(key_tr).mean()     # 올바른 방법: 학습용으로만 계산
rate_leak = df["target"].groupby(day_key).mean()        # 누수: 평가용 정답까지 포함해 계산

print("올바른 방법 (학습용 가입률):", round(day_rate_auc(rate_ok), 4))
print("누수 방법   (전체 가입률)  :", round(day_rate_auc(rate_leak), 4))

# 참고: 통화가 적은 날일수록 누수가 커지는지 확인
cnt = day_key.value_counts()
print()
print("통화 10건 미만인 날:", int((cnt < 10).sum()), "일 /", cnt.size, "일")

올바른 방법 (학습용 가입률): 0.7287
누수 방법   (전체 가입률)  : 0.7481

통화 10건 미만인 날: 60 일 / 318 일


> **[판단]** 이번에는 두 방법의 점수가 다른가요? 2-1의 스케일러와 무엇이 다르기에 이번에는 차이가 났을까요? "그날 통화가 몇 건뿐인 날"이라면 누수가 더 커질까요, 작아질까요?
>
> 나의 답: 이번에는 **다릅니다**. 올바른 방법 0.7287, 누수 방법 0.7481로 약 0.02 차이가 납니다. 2-1의 스케일러는 피처의 평균·표준편차만 썼지만, 타깃 인코딩은 **평가용 고객의 정답(가입 여부)**을 피처 값 안에 넣습니다. 그래서 모델이 평가용 정답을 간접적으로 보고 답하는 셈이 됩니다. 그날 통화가 몇 건뿐인 날이라면 **누수가 더 커집니다.** 예를 들어 통화가 3건인 날에 평가용 고객 1명이 가입했다면, 그날 가입률에 그 고객의 정답이 1/3이나 반영됩니다. 통화가 많은 날은 한 명의 영향이 묽어집니다. 이 데이터에는 통화 10건 미만인 날이 318일 중 60일이나 됩니다.

## 2-3. `get_dummies`의 함정

운영에서는 하루치 고객이 조금씩 들어옵니다. **[제공]** 셀은 평가용 고객 중 20명만 뽑아 `get_dummies`로 바꾸고, 학습할 때의 열과 비교합니다.

In [10]:
# [제공] 새 고객 20명이 들어왔다면
new_customers = test_df.sample(20, random_state=1)
train_cols = pd.get_dummies(df[features]).columns
new_cols = pd.get_dummies(new_customers[features]).columns
print("학습할 때 열 수:", len(train_cols), "| 새 고객 20명의 열 수:", len(new_cols))
print("새 고객에게 없는 열 (예시 10개):", sorted(set(train_cols) - set(new_cols))[:10])

학습할 때 열 수: 46 | 새 고객 20명의 열 수: 38
새 고객에게 없는 열 (예시 10개): ['job_entrepreneur', 'job_self-employed', 'job_student', 'job_unknown', 'month_aug', 'month_jan', 'month_sep', 'poutcome_success']


> **[판단]** 열 수가 다르면 학습한 모델에 새 고객을 넣을 때 어떤 일이 생길까요? 지금까지 Chapter 1~3에서 이 문제를 겪지 않은 이유는 무엇일까요?
>
> 나의 답: 열 수가 다르면(학습 46열, 새 고객 38열) 모델이 기대하는 입력 모양과 맞지 않아 **오류가 나거나**, 열 이름을 무시하고 위치로 넣으면 **엉뚱한 열에 엉뚱한 계수가 곱해져** 조용히 잘못된 예측이 나옵니다. 오류보다 조용히 틀리는 쪽이 더 위험합니다. Chapter 1~3에서 이 문제를 겪지 않은 이유는 **나누기 전에 전체 데이터로 한 번에 `get_dummies`를 했기 때문**입니다. 학습용과 평가용이 같은 열 목록을 공유했지만, 이것도 평가용에 어떤 범주가 있는지 미리 본 셈이라 운영에서는 쓸 수 없는 방법입니다.

## 2-4. 파이프라인으로 다시 만들기

**[직접 작성]** `ColumnTransformer`와 `Pipeline`으로 Chapter 3 모델을 다시 만드세요.

1. `preprocess`: 숫자 열(`num_cols`)에는 `StandardScaler()`, 범주 열(`cat_cols`)에는 `OneHotEncoder(handle_unknown="ignore")`
2. `pipe_model`: `Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])`
3. **학습용 데이터 그대로**(`train_df[features]`) `fit`하고, 평가용 확률을 `prob_pipe`에 저장해 `record("파이프라인", y_test, prob_pipe)`로 기록
4. 2-3의 `new_customers`에도 `predict_proba`가 동작하는지 확인

<details><summary>힌트</summary>

```python
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])
pipe_model.fit(train_df[features], y_train)
```
</details>

In [11]:
# 2-4. ColumnTransformer + Pipeline으로 다시 만들기
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])
pipe_model.fit(train_df[features], y_train)        # 원본 학습용 데이터를 그대로 넣는다

prob_pipe = pipe_model.predict_proba(test_df[features])[:, 1]
record("파이프라인", y_test, prob_pipe)

# 새 고객 20명에게도 그대로 동작하는지 확인
new_prob = pipe_model.predict_proba(new_customers[features])[:, 1]
print("새 고객 20명 예측 확률 (앞 5명):", new_prob[:5].round(3), "| 개수:", len(new_prob))

기록: [파이프라인 / 무작위] ROC AUC 0.773 | 상위 1,000명 가입 465명 (무작위의 4.0배)
새 고객 20명 예측 확률 (앞 5명): [0.22  0.18  0.117 0.076 0.578] | 개수: 20


> **[판단]** 파이프라인의 점수는 1-2의 "손으로 순서 맞추기"와 같은가요? 점수가 같다면, 파이프라인으로 바꿔서 얻은 것은 무엇인가요? 2-1, 2-3과 연결해 적어 보세요.
>
> 나의 답: 점수는 0.773으로 같습니다(상위 1,000명 가입 464명 → 465명, 사실상 같음). 점수는 같지만 얻은 것은 **안전성**입니다. ① 스케일러의 기준이 `fit`할 때 들어온 학습용 데이터로만 자동으로 잡혀서 2-1 같은 실수를 할 수 없습니다. ② `OneHotEncoder(handle_unknown="ignore")`가 학습 때의 열 목록을 기억하므로, 2-3처럼 범주가 일부 빠진 20명이 들어와도 그대로 예측됩니다. ③ 전처리와 모델이 한 덩어리라서 교차검증이나 운영에 통째로 넘길 수 있습니다.

---
# Part 3. 점수를 믿는 법: 교차검증과 시간 순 분할

## 3-1. 5겹 교차검증

Chapter 1~3에서는 평가용 점수를 보며 설정(깊이, $\alpha$, 임계값)을 골랐습니다. 이제는 설정을 고르는 데 **교차검증**을 쓰고, 평가용은 마지막 확인에만 씁니다.

**[직접 작성]** 2-4의 `pipe_model`을 **학습용 데이터로** 5겹 교차검증하세요. 점수 다섯 개와 평균, 표준편차를 출력합니다.

<details><summary>힌트</summary>

```python
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_model, train_df[features], y_train, cv=cv, scoring="roc_auc")
```
</details>

In [12]:
# 3-1. 학습용 데이터로 5겹 교차검증
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_model, train_df[features], y_train, cv=cv, scoring="roc_auc")
print("fold별 ROC AUC:", scores.round(4))
print(f"평균 {scores.mean():.4f} | 표준편차 {scores.std():.4f}")
print(f"평가용 ROC AUC: {roc_auc_score(y_test, prob_pipe):.4f}")

fold별 ROC AUC: [0.7647 0.7645 0.7595 0.7687 0.762 ]
평균 0.7639 | 표준편차 0.0031
평가용 ROC AUC: 0.7729


> **[판단]** 교차검증 평균은 평가용 ROC AUC(0.773)와 얼마나 가깝나요? 교차검증에는 평가용 데이터가 쓰였나요? 파이프라인을 통째로 넣었다는 점은 왜 중요할까요?
>
> 나의 답: 교차검증 평균은 0.764(표준편차 0.003)로, 평가용 0.773보다 약 0.009 낮습니다. 차이가 fold 편차의 3배 정도라서, 평가용 점수가 이번 분할에서 조금 운이 좋았을 수 있습니다. 그래도 둘 다 0.76~0.77 수준이라 크게 어긋나지는 않습니다. 교차검증에는 **평가용 데이터를 쓰지 않았습니다**(`train_df`만 사용). 파이프라인을 통째로 넣었기 때문에 각 fold마다 스케일러와 인코더도 **그 fold의 학습 부분으로만** 다시 `fit`됩니다. 전처리를 미리 해 두고 모델만 넣었다면 검증 fold의 정보가 전처리 기준에 섞여 2-1, 2-2의 누수가 교차검증 안에서 다시 생깁니다.

## 3-2. 두 모델을 비교할 때: 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 세 쌍의 모델을 같은 5겹 교차검증으로 비교합니다.

| 비교 | 모델 A | 모델 B |
| --- | --- | --- |
| ① | 로지스틱 회귀 ($C = 1$, 기본값) | 로지스틱 회귀 ($C = 0.01$, 규제를 강하게) |
| ② | 결정트리 깊이 5 | 결정트리 깊이 8 |
| ③ | 로지스틱 회귀 ($C = 1$) | 결정트리 깊이 8 |

> **[판단]** 예상: 세 비교 중 "B가 A보다 낫다(또는 못하다)"고 자신 있게 보고할 수 있는 비교는 어느 것일까요? 이유도 적어 보세요.
>
> 나의 답: ③(로지스틱 회귀 vs 결정트리 깊이 8)만 자신 있게 보고할 수 있을 것 같습니다. 서로 다른 종류의 모델이라 차이가 클 것이고, Chapter 3에서도 로지스틱 회귀가 트리보다 확실히 좋았습니다. ①은 같은 모델의 규제 강도만 바꾼 것이라 차이가 매우 작을 것이고, ②는 트리 깊이에 따라 점수가 fold마다 들쭉날쭉해서 판단하기 어려울 것 같습니다.

## 3-3. fold별 차이로 비교하기

**[제공]** 모델을 쉽게 만드는 함수 `make_pipe`와, 두 모델을 **같은 5겹**으로 평가해 fold별 점수와 차이를 보여 주는 함수 `compare`입니다.

In [13]:
# [제공] 모델 만들기와 대응 비교
def make_pipe(clf, scale=True):
    pre = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("pre", pre), ("clf", clf)])

def compare(name_a, model_a, name_b, model_b):
    s_a = cross_val_score(model_a, train_df[features], y_train, cv=cv, scoring="roc_auc")
    s_b = cross_val_score(model_b, train_df[features], y_train, cv=cv, scoring="roc_auc")
    diff = s_b - s_a
    print(f"{name_a}: {s_a.round(3)} | 평균 {s_a.mean():.3f} ± {s_a.std():.3f}")
    print(f"{name_b}: {s_b.round(3)} | 평균 {s_b.mean():.3f} ± {s_b.std():.3f}")
    print(f"fold별 차이 (B - A): {diff.round(3)} | 평균 차이 {diff.mean():+.3f}")
    print()

**[직접 작성]** `make_pipe`로 모델 네 개를 만들고, `compare`로 3-2의 세 쌍을 비교하세요. 결정트리는 스케일링이 필요 없으므로 `scale=False`로 만듭니다(교안 2-6). 트리는 `random_state=42`를 줍니다.

<details><summary>힌트</summary>

```python
logit_1 = make_pipe(LogisticRegression(max_iter=3000))
logit_001 = make_pipe(LogisticRegression(max_iter=3000, C=0.01))
tree_5 = make_pipe(DecisionTreeClassifier(max_depth=5, random_state=42), scale=False)
compare("C=1", logit_1, "C=0.01", logit_001)
```
</details>

In [14]:
# 3-3. 모델 네 개를 만들고 세 쌍 비교
logit_1 = make_pipe(LogisticRegression(max_iter=3000))
logit_001 = make_pipe(LogisticRegression(max_iter=3000, C=0.01))
tree_5 = make_pipe(DecisionTreeClassifier(max_depth=5, random_state=42), scale=False)
tree_8 = make_pipe(DecisionTreeClassifier(max_depth=8, random_state=42), scale=False)

print("비교 ①"); compare("로지스틱 C=1", logit_1, "로지스틱 C=0.01", logit_001)
print("비교 ②"); compare("트리 깊이 5", tree_5, "트리 깊이 8", tree_8)
print("비교 ③"); compare("로지스틱 C=1", logit_1, "트리 깊이 8", tree_8)

비교 ①


로지스틱 C=1: [0.765 0.765 0.759 0.769 0.762] | 평균 0.764 ± 0.003
로지스틱 C=0.01: [0.763 0.76  0.759 0.765 0.761] | 평균 0.761 ± 0.002
fold별 차이 (B - A): [-0.002 -0.005 -0.    -0.004 -0.001] | 평균 차이 -0.002

비교 ②


트리 깊이 5: [0.688 0.713 0.7   0.691 0.699] | 평균 0.698 ± 0.009
트리 깊이 8: [0.705 0.715 0.711 0.693 0.711] | 평균 0.707 ± 0.008
fold별 차이 (B - A): [0.016 0.002 0.011 0.002 0.012] | 평균 차이 +0.009

비교 ③


로지스틱 C=1: [0.765 0.765 0.759 0.769 0.762] | 평균 0.764 ± 0.003
트리 깊이 8: [0.705 0.715 0.711 0.693 0.711] | 평균 0.707 ± 0.008
fold별 차이 (B - A): [-0.06  -0.05  -0.048 -0.076 -0.051] | 평균 차이 -0.057



> **[판단]** 비교 ①: fold별 차이의 방향은 일관되나요? 크기는 실무적으로 의미가 있나요? 둘 중 무엇을 고르겠습니까?
>
> 나의 답: C=0.01이 다섯 fold **모두에서 조금씩 낮아** 방향은 일관됩니다(차이 −0.000 ~ −0.005). 하지만 평균 차이가 −0.002로, fold 표준편차(0.003) 수준이라 **실무적으로 의미 있는 차이는 아닙니다.** 상위 고객 순위가 거의 같을 것이므로 어느 쪽이든 결과는 비슷합니다. 고른다면 기본값 **C=1**을 쓰겠습니다. 조금이라도 높고, 굳이 바꿀 근거가 없기 때문입니다.

> **[판단]** 비교 ②: 다섯 번 모두 같은 쪽이 이겼나요? 차이의 크기는 fold마다 어떻게 다른가요? "깊이 8이 낫다"고 보고하겠습니까?
>
> 나의 답: 다섯 번 모두 깊이 8이 이겼습니다. 하지만 차이 크기는 fold마다 +0.002 ~ +0.016으로 8배나 차이가 납니다. 평균 차이 +0.009는 각 모델의 fold 표준편차(0.008~0.009)와 비슷한 크기입니다. "깊이 8이 **약간** 낫다"고는 말할 수 있지만, 두 트리 모두 로지스틱 회귀(0.764)보다 크게 낮아서 보고할 의미가 거의 없습니다. 깊이 8을 정말 쓰려면 깊이 6, 7, 10 등도 함께 교차검증으로 확인해야 합니다.

> **[판단]** 비교 ③: 차이가 표준편차의 몇 배쯤 되나요? 이 비교는 "낫다"고 보고할 수 있나요? 3-2의 예상은 맞았나요?
>
> 나의 답: 평균 차이는 **−0.057**로, 로지스틱 회귀의 표준편차(0.003)의 약 19배, 트리의 표준편차(0.008)의 약 7배입니다. 다섯 fold 모두에서 0.048 이상 차이가 나므로 "로지스틱 회귀가 트리 깊이 8보다 낫다"고 **자신 있게 보고할 수 있습니다.** 예상은 맞았습니다. ①은 예상대로 차이가 거의 없었고, ②는 방향은 일관됐지만 크기가 편차 수준이라 확신하기 어려웠습니다.

## 3-4. 교차검증으로 설정 고르기

**[직접 작성]** `GridSearchCV`로 로지스틱 회귀의 `C`를 0.01, 0.1, 1, 10 중에서 고르세요. 가장 좋은 설정(`best_params_`)과, 후보별 교차검증 평균·표준편차 표를 출력합니다.

<details><summary>힌트</summary>

파이프라인 안의 설정은 `단계이름__설정이름`으로 가리킵니다.
```python
search = GridSearchCV(make_pipe(LogisticRegression(max_iter=3000)), {"clf__C": [0.01, 0.1, 1, 10]},
                      cv=cv, scoring="roc_auc")
search.fit(train_df[features], y_train)
pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]]
```
</details>

In [15]:
# 3-4. GridSearchCV로 C 고르기 (학습용 데이터 + 교차검증만 사용)
search = GridSearchCV(make_pipe(LogisticRegression(max_iter=3000)), {"clf__C": [0.01, 0.1, 1, 10]},
                      cv=cv, scoring="roc_auc")
search.fit(train_df[features], y_train)
print("가장 좋은 설정:", search.best_params_, "| 교차검증 평균", round(search.best_score_, 4))

pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]].round(4)

가장 좋은 설정: {'clf__C': 0.1} | 교차검증 평균 0.7641


,param_clf__C,mean_test_score,std_test_score
0,0.01,0.7615,0.0021
1,0.10,0.7641,0.0029
2,1.00,0.7639,0.0031
3,10.00,0.7638,0.0031


> **[판단]** `GridSearchCV`가 고른 설정과 다른 후보들의 차이는 3-3의 기준으로 구별할 만한 크기인가요? 여러분이라면 어떤 `C`를 쓰겠습니까?
>
> 나의 답: C=0.1(0.7641), C=1(0.7639), C=10(0.7638)의 차이는 0.0003 이하로, 표준편차(약 0.003)의 10분의 1밖에 안 됩니다. 3-3의 기준으로는 **구별할 수 없는 차이**입니다. C=0.01(0.7615)만 조금 낮습니다. 저라면 GridSearchCV가 고른 0.1을 그대로 쓰거나 기본값 1을 쓰겠습니다. 어느 쪽이든 결과는 같으므로, "C를 튜닝해서 좋아졌다"고 보고하지는 않겠습니다.

## 3-5. 시간이 있는 데이터는 시간 순으로

이 파일의 행은 2008년 5월부터 2010년 11월까지 **날짜순**으로 정렬되어 있습니다. **[제공]** 셀은 행을 순서대로 열 묶음으로 나눠 가입률을 보여 줍니다.

In [16]:
# [제공] 시간 순서대로 본 가입률
deciles = pd.Series(np.arange(len(df)) * 10 // len(df), index=df.index)
print(df.groupby(deciles)["target"].mean().round(3).rename("가입률").rename_axis("구간 (0 = 처음 10%)"))

구간 (0 = 처음 10%)
0    0.030
1    0.037
2    0.048
3    0.064
4    0.061
5    0.062
6    0.105
7    0.131
8    0.161
9    0.471
Name: 가입률, dtype: float64


**[직접 작성]** 앞 80%의 행으로 학습하고 뒤 20%의 행으로 평가하는 **시간 순 분할**을 만드세요. 2-4와 같은 파이프라인을 학습용(앞 80%)으로 `fit`하고, 평가용(뒤 20%)의 확률을 `prob_time`에 저장한 뒤 `record("Chapter 3 모델", y_test_time, prob_time, split="시간 순")`으로 기록하세요.

<details><summary>힌트</summary>

```python
cut = int(len(df) * 0.8)
train_time, test_time = df.iloc[:cut], df.iloc[cut:]
y_test_time = test_time["target"]
```
`make_pipe(LogisticRegression(max_iter=3000))`로 새 파이프라인을 만들어 `train_time[features]`로 `fit`합니다.
</details>

In [17]:
# 3-5. 시간 순 분할: 앞 80%로 학습, 뒤 20%로 평가
cut = int(len(df) * 0.8)
train_time, test_time = df.iloc[:cut], df.iloc[cut:]
y_train_time, y_test_time = train_time["target"], test_time["target"]
print(f"학습용 {len(train_time):,}건 (가입률 {y_train_time.mean():.3f}) | 평가용 {len(test_time):,}건 (가입률 {y_test_time.mean():.3f})")

pipe_time = make_pipe(LogisticRegression(max_iter=3000)).fit(train_time[features], y_train_time)
prob_time = pipe_time.predict_proba(test_time[features])[:, 1]
record("Chapter 3 모델", y_test_time, prob_time, split="시간 순")

학습용 36,168건 (가입률 0.067) | 평가용 9,043건 (가입률 0.316)
기록: [Chapter 3 모델 / 시간 순] ROC AUC 0.692 | 상위 1,000명 가입 532명 (무작위의 1.7배)


> **[판단]** 시간 순으로 나누면 ROC AUC가 얼마나 달라지나요? 왜 이런 차이가 생길까요? 실제 캠페인과 더 가까운 조건은 어느 쪽인가요?
>
> 나의 답: ROC AUC가 0.773(무작위) → **0.692**(시간 순)로 약 0.08 떨어집니다. 이유는 시간에 따라 상황이 바뀌기 때문입니다. 가입률이 처음 10%에서는 3.0%였지만 마지막 10%에서는 47.1%로 크게 올랐습니다. 학습용(앞 80%) 가입률은 6.7%인데 평가용(뒤 20%)은 31.6%입니다. 또 뒤 기간에는 월 구성, 고객층, 이전 캠페인 결과도 달라서 과거에 배운 패턴이 그대로 통하지 않습니다. 무작위 분할은 미래 고객이 학습용에 섞여 "미래를 조금 보고" 시험을 친 셈입니다. 실제 캠페인은 **과거 데이터로 배운 모델을 미래 고객에게 쓰는 것**이므로 **시간 순 분할**이 더 가까운 조건입니다.

> **[판단]** 그런데 시간 순 분할의 "상위 1,000명 가입"은 무작위 분할보다 많거나 적은가요? ROC AUC는 낮아졌는데 이 숫자는 왜 이렇게 나왔을까요? 결과표의 "평가 가입률"과 "배수" 열을 함께 보세요.
>
> 나의 답: 시간 순 분할의 상위 1,000명 가입은 **532명**으로, 무작위 분할(464명)보다 **많습니다.** 그런데 이것은 모델이 좋아져서가 아니라 **평가 기간의 가입률 자체가 높아서**입니다. 평가 가입률이 11.7% → 31.6%로 약 2.7배 높아서, 아무나 1,000명을 골라도 316명이 가입합니다. 그래서 "배수"는 4.0배 → **1.7배**로 크게 떨어졌습니다. 상위 1,000명 가입 수는 모델 실력과 그 기간의 가입률이 섞인 숫자이므로, 가입률이 다른 기간끼리 이 숫자를 그대로 비교하면 안 됩니다.

---
# Part 4. 보고할 숫자 정하기

이제 질문으로 돌아옵니다. **마케팅팀에 "다음 캠페인에서 기대할 가입 수"로 보고할 숫자는 어느 것인가?**

## 4-1. 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 네 가지 조합의 점수를 한 표에 놓습니다.

| | 무작위 분할 | 시간 순 분할 |
| --- | --- | --- |
| 통화 시간 없음 | 1-2 (기록됨) | 3-5 (기록됨) |
| 통화 시간 있음 | 1-3 (기록됨) | 4-2에서 만듦 |

> **[판단]** 예상: 네 조합 중 ROC AUC가 가장 높은 것과 가장 낮은 것은 무엇일까요? 다음 캠페인에서 실제로 나올 성과에 가장 가까운 것은 어느 칸일까요?
>
> 나의 답: 가장 높은 것은 **통화 시간 있음 + 무작위 분할**, 가장 낮은 것은 **통화 시간 없음 + 시간 순 분할**일 것 같습니다. 다음 캠페인 성과에 가장 가까운 칸은 **통화 시간 없음 + 시간 순 분할**입니다. 전화 전에 알 수 있는 정보만 쓰고, 과거로 배워 미래를 예측하는 실제 상황과 같기 때문입니다.

## 4-2. 네 조합 비교하기

**[직접 작성]** 남은 한 칸(통화 시간 있음 + 시간 순 분할)을 만드세요. `features + ["duration"]`을 쓰는 파이프라인을 만들어야 하므로, `make_pipe`의 숫자 열 목록에 `duration`을 더해야 합니다. 확률을 기록하고 `show_log()`로 결과표 전체를 보세요.

<details><summary>힌트</summary>

`make_pipe`는 `num_cols`를 씁니다. 이 셀에서만 직접 만들면 됩니다.
```python
pre_dur = ColumnTransformer([
    ("num", StandardScaler(), num_cols + ["duration"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
```
</details>

In [18]:
# 4-2. 통화 시간 있음 + 시간 순 분할
pre_dur = ColumnTransformer([
    ("num", StandardScaler(), num_cols + ["duration"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_dur_time = Pipeline([("pre", pre_dur), ("clf", LogisticRegression(max_iter=3000))])
pipe_dur_time.fit(train_time[features + ["duration"]], y_train_time)
prob_dur_time = pipe_dur_time.predict_proba(test_time[features + ["duration"]])[:, 1]
record("통화 시간 추가", y_test_time, prob_dur_time, split="시간 순")

# 참고: Chapter 3의 현행 규칙(이전 캠페인 성공 또는 60세 초과)과 같은 인원으로 비교
for split, frame, prob in [("무작위", test_df, prob_pipe), ("시간 순", test_time, prob_time)]:
    rule = ((frame["poutcome"] == "success") | (frame["age"] > 60)).values
    y = frame["target"].values
    top = y[np.argsort(-prob, kind="stable")][:rule.sum()].sum()
    print(f"[{split}] 규칙이 고른 {rule.sum():,}명 중 가입 {y[rule].sum():,}명 | 모델 상위 {rule.sum():,}명 중 가입 {top:,}명")

show_log()

기록: [통화 시간 추가 / 시간 순] ROC AUC 0.782 | 상위 1,000명 가입 598명 (무작위의 1.9배)


[무작위] 규칙이 고른 484명 중 가입 265명 | 모델 상위 484명 중 가입 282명
[시간 순] 규칙이 고른 2,056명 중 가입 1,171명 | 모델 상위 2,056명 중 가입 1,163명


,실험,분할,ROC AUC,PR AUC,평가 가입률,"상위 1,000명 가입","무작위 1,000명 기대",배수
0,Chapter 3 모델,무작위,0.773,0.411,0.117,464,117,4.0
1,통화 시간 추가,무작위,0.906,0.544,0.117,578,117,4.9
2,파이프라인,무작위,0.773,0.411,0.117,465,117,4.0
3,Chapter 3 모델,시간 순,0.692,0.496,0.316,532,316,1.7
4,통화 시간 추가,시간 순,0.782,0.569,0.316,598,316,1.9


> **[판단]** 예상은 맞았나요? 네 조합 중 마케팅팀에 보고할 점수는 어느 칸이고, 왜 그런가요? 통화 시간을 넣은 두 칸은 왜 보고할 수 없나요?
>
> 나의 답: 예상이 맞았습니다. ROC AUC는 통화 시간 있음·무작위 0.906 > 통화 시간 있음·시간 순 0.782 > 통화 시간 없음·무작위 0.773 > 통화 시간 없음·시간 순 0.692 순입니다. 보고할 칸은 **통화 시간 없음 + 시간 순 분할**(ROC AUC 0.692, 무작위로 고른 것의 1.7배)입니다. 실제로 쓸 수 있는 정보만 쓰고, 실제와 같은 "과거 → 미래" 조건에서 쟀기 때문입니다. 통화 시간을 넣은 두 칸은 전화를 걸기 전에는 존재하지 않는 값을 쓴 점수라서, 실제 캠페인에서는 그 모델을 **아예 돌릴 수 없습니다.** 점수가 높아도 실제로 재현될 수 없는 숫자입니다.

> **[판단]** 마케팅팀은 "1,000통이면 몇 명이 가입하나요?"라고 묻습니다. 시간 순 분할의 "상위 1,000명 가입" 숫자를 그대로 약속해도 될까요? 다음 캠페인의 가입률이 지난 기간과 다를 수 있다는 점을 생각하면, 어떤 형태로 말하는 것이 정직할까요?
>
> 나의 답: **그대로 약속하면 안 됩니다.** 532명은 평가 기간의 가입률이 31.6%로 매우 높았기 때문에 나온 숫자입니다. 다음 캠페인의 가입률이 학습 기간처럼 6~7%라면 가입 수는 훨씬 적을 것이고, 가입률이 더 높아질 수도 있습니다. 정직한 형태는 **상대적인 숫자와 범위**입니다. 예: "모델이 고른 1,000명은 아무나 1,000명을 고를 때보다 가입자가 약 1.7배(시간 순 기준) 많을 것으로 봅니다. 가입 수 자체는 그때의 전체 가입률에 따라 달라집니다. 전체 가입률이 10%라면 약 170명, 30%라면 약 500명 정도입니다." 그리고 캠페인 초기 일부 결과로 가입률을 확인한 뒤 기대치를 다시 알려 드리겠다고 덧붙입니다.

## 4-3. 점검표 채우기

교안 3-6의 여섯 질문을 **보고하려는 모델**(통화 시간 없음, 파이프라인)에 대해 하나씩 답하세요. 각 질문에 "통과 / 미통과 / 확인 못 함"과 근거를 적습니다.

> **[판단]** 점검표:
>
> ① 쓴 변수가 모두 예측 시점에 존재하는가: **확인 못 함(대부분 통과).** `duration`은 뺐습니다. 하지만 `campaign`(마지막 통화 포함), `month`(마지막 통화의 월), `contact`는 통화 뒤에 정해지는 부분이 있어서, 데이터를 만든 부서에 기록 시점을 확인해야 합니다.
>
> ② 전처리 기준을 학습용으로만 잡았는가: **통과.** 파이프라인으로 스케일러와 원핫 인코더를 학습용으로만 `fit`했고, 교차검증 안에서도 fold마다 다시 `fit`됩니다. 타깃 인코딩은 쓰지 않았습니다.
>
> ③ 설정을 고를 때 평가용 데이터를 쓰지 않았는가: **통과.** `C`는 학습용 데이터의 5겹 교차검증(GridSearchCV)으로 골랐습니다. 다만 Chapter 3에서는 평가용 점수를 보며 피처와 설정을 골랐으므로, 그 부분은 평가용을 일부 본 셈입니다.
>
> ④ 비교한 차이가 fold 편차보다 크고 의미 있는가: **통과.** 로지스틱 회귀와 트리의 차이(0.057)는 표준편차의 7배 이상입니다. 반면 C 값끼리의 차이(0.0003 이하)는 편차보다 작으므로 "튜닝으로 좋아졌다"고 주장하지 않습니다.
>
> ⑤ 평가용이 학습용보다 나중인가: **통과(시간 순 분할 기준).** 보고하는 점수는 앞 80%로 학습하고 뒤 20%로 평가한 것입니다. 무작위 분할 점수(0.773)는 이 조건을 통과하지 못하므로 보고하지 않습니다.
>
> ⑥ 베이스라인과 현행 규칙보다 얼마나 나은가: **부분 통과.** 무작위로 고르는 것(베이스라인)보다는 시간 순 분할에서도 1.7배 낫습니다. 그런데 Chapter 3의 현행 규칙(이전 캠페인 성공 또는 60세 초과)과 같은 인원으로 비교하면(4-2 셀 출력), 무작위 분할에서는 모델이 조금 낫지만(484명 중 282명 vs 265명), 시간 순 분할에서는 **비슷합니다**(2,056명 중 1,163명 vs 1,171명). 미래 기간에는 모델이 현행 규칙보다 낫다고 말할 수 없습니다.
>
> 나의 답: 보고하려는 모델은 누수 점검(①②③)과 비교 기준(④⑤)은 대체로 통과했습니다. 다만 `campaign`, `month`의 기록 시점은 확인하지 못했고, 시간 순 조건에서는 현행 규칙보다 낫다는 근거가 없습니다(⑥). 그래서 모델의 강점은 "현행 규칙보다 많이 맞힌다"가 아니라 "규칙과 비슷한 성과를 내면서, 원하는 인원 수(1,000명, 2,000명 등)에 맞춰 가능성 높은 순서로 고를 수 있다"로 보고해야 합니다.

## 4-4. 마케팅팀 보고 메모

분석 결과를 마케팅팀장에게 메시지로 보낸다고 생각하고 쓰세요. 팀장은 머신러닝 용어를 모릅니다. 아래 네 가지가 모두 들어가야 합니다.

1. **보고하는 숫자와 그 근거**: 어떤 조건(분할 방식)에서 잰 숫자인지
2. **통화 시간 모델을 쓰지 않는 이유**: 점수가 더 높은데도 쓰지 않는 이유를 쉬운 말로
3. **1,000통의 기대 효과**: 절대 숫자를 약속할 수 있는지, 아니라면 어떤 형태로 말하는지
4. **주의할 점**: 다음 캠페인의 가입률이 달라질 가능성, 아직 확인하지 못한 것

> **[판단]** 마케팅팀 보고 메모:
>
> 1. 보고하는 숫자와 근거: 다음 캠페인에서 기대할 효과로 **"모델이 고른 고객은 아무나 고른 고객보다 가입자가 약 1.7배 많다"**를 보고드립니다. 과거 기간(앞 80%) 데이터만으로 모델을 만들고, 그 뒤 기간(뒤 20%, 9,043건)의 실제 결과로 확인한 숫자입니다. "과거로 배워 미래에 쓴다"는 실제 캠페인과 같은 조건이라 가장 믿을 만합니다. 데이터를 무작위로 섞어서 재면 4.0배가 나오지만, 미래를 일부 미리 보고 잰 숫자라서 보고하지 않습니다.
>
> 2. 통화 시간 모델을 쓰지 않는 이유: 통화 시간을 넣은 모델의 점수가 더 높은 것은 사실입니다. 하지만 통화 시간은 **전화가 끝나야 알 수 있는 정보**입니다. 관심 있는 고객이 오래 통화하는 것이라 "이미 결과를 보고 맞힌" 것과 같습니다. 누구에게 전화할지 정하는 시점에는 이 값이 없어서, 실제 캠페인에서는 이 모델을 쓸 수 없습니다.
>
> 3. 1,000통의 기대 효과: 정확한 가입자 수를 약속드리기는 어렵습니다. 가입 수는 모델뿐 아니라 그 시기 전체 고객의 가입률에 크게 좌우되기 때문입니다(과거 기록에서도 시기별 가입률이 3%에서 47%까지 달랐습니다). 대신 이렇게 말씀드릴 수 있습니다. 1,000명에게 전화한다면, 전체 가입률이 10% 정도일 때 약 170명, 30% 정도일 때 약 500명 정도를 기대할 수 있습니다. 아무나 1,000명에게 걸 때(각각 100명, 300명)보다 많습니다.
>
> 4. 주의할 점: ① 다음 캠페인의 가입률이 지난 기간과 다르면 가입 수도 달라집니다. 캠페인 초기 1~2주의 결과로 가입률을 확인하고 기대치를 다시 알려 드리겠습니다. ② 최근 기간에서는 기존 규칙("이전 캠페인 성공 고객 또는 60세 초과 고객")과 모델의 성과가 비슷했습니다. 모델의 장점은 원하는 인원 수에 맞춰 가능성 높은 순서로 고객을 고를 수 있다는 점입니다. ③ 일부 항목(이번 캠페인 통화 횟수, 통화 월)이 전화 전에 확정되는 값인지 아직 확인하지 못했습니다. 확인 후 필요하면 모델을 고치겠습니다.
>
> 나의 답: 위 메모와 같습니다. 핵심은 "과거로 배워 미래에 쓴 조건에서, 아무나 고를 때보다 약 1.7배"이며, 절대 가입자 수는 그때의 가입률에 따라 범위로 말씀드립니다.

---
# 마무리

## 이 Chapter의 판단 포인트

> **이 정보는 예측 시점에 알 수 있는가? 그리고 두 모델의 점수 차이는 교차검증 편차보다 큰가?**

## 개인 검증 체크리스트에 추가

오늘 실습에서 내가 실제로 겪은 일을 바탕으로, 체크리스트의 줄을 내 말로 적으세요.

예: "점수가 갑자기 좋아진 변수를 보면, 그 값이 언제 기록되는지부터 확인한다."

> **[판단]** 나의 체크리스트 6번(누수를 점검할 때):
>
> 나의 답: 점수가 갑자기 크게 좋아진 변수를 보면, 그 값이 **언제 기록되는지**(전화 전인지 후인지)부터 확인한다. 그리고 스케일러·인코더 같은 전처리는 **파이프라인에 넣어** 학습용(교차검증에서는 각 fold의 학습 부분)으로만 기준을 잡는다. 특히 정답을 쓰는 전처리(타깃 인코딩)는 평가용 정답이 섞이지 않았는지 반드시 확인한다.

> **[판단]** 나의 체크리스트 7번(점수를 비교하고 보고할 때):
>
> 나의 답: 두 모델을 비교할 때는 평균만 보지 말고 **같은 fold끼리의 차이**를 보고, 그 차이가 fold 표준편차보다 확실히 클 때만 "낫다"고 말한다. 데이터에 시간이 있으면 **시간 순 분할**로 잰 점수를 보고하고, 가입 수 같은 절대 숫자 대신 **무작위·현행 규칙 대비 배수**와 가입률에 따른 범위로 말한다.

## 제출물

- 이 노트북 (모든 **[판단]** 칸을 채운 상태)

수고했습니다. 다음 Chapter 5에서는 트리를 여러 개 모아 더 강한 모델을 만들고, 그 모델이 무엇을 보고 예측하는지 SHAP으로 설명합니다.